# Neuron Clustering

This notebook clusters neurons by subclass using cosine-distance k-means on MLP activations.

## Imports and Setup

In [ ]:
import json
import os
import sys

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer

# Add src to path for imports
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from utils import load_model_checkpoint, _stack_layer_activations, _safe_model_name
from circuit_discovery.utils import parse_equation
from gen_activations_dataset import NeuronActivationsGenerator

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Configuration

Set your model name and checkpoint path here.

In [ ]:
# Configuration - modify these as needed
model_name = "meta-llama/Llama-3.2-1B"  # or "meta-llama/Meta-Llama-3-8B"
checkpoint_path = "epoch_4000.pt"  # Path to your checkpoint file
k_classes = 8
lr = 1e-3
threshold = 1e-3

## Load Model and Tokenizer

In [ ]:
# Load the model from checkpoint
model, optimizer, metrics_log, epoch = load_model_checkpoint(
    checkpoint_path, 
    k_classes=k_classes, 
    lr=lr
)
model.eval()
print(f"Loaded checkpoint from epoch {epoch}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded for {model_name}")

## Extract Neuron Masks

In [ ]:
# Get neuron masks based on model
if model_name == "meta-llama/Llama-3.2-1B":
    neuron_masks = model.neuron_masks_1b.class_masks()
else:
    neuron_masks = model.neuron_masks_8b.class_masks()

neuron_masks = neuron_masks > (1 - threshold)

print(f"Active neurons ratio: {torch.mean(torch.mean(neuron_masks.float(), dim=1)).item():.4f}")
print(f"\nNeurons per subclass:")
for i in range(k_classes):
    count = neuron_masks[i].count_nonzero().item()
    print(f"  Subclass {i}: {count} neurons")

## K-Means Clustering (Cosine Distance)

In [ ]:
def _kmeans_cosine(x, k, num_iters=20):
    """
    Balanced k-means clustering using cosine distance.
    
    Args:
        x: Input tensor of shape (N, D)
        k: Number of clusters
        num_iters: Maximum number of iterations
    
    Returns:
        cluster_ids: Cluster assignment for each point
        centroids: Final cluster centroids
        loss: Mean cosine distance to assigned centroids
    """
    N, D = x.shape
    if k > N:
        raise ValueError("k cannot be larger than number of points")

    x = F.normalize(x, p=2, dim=-1, eps=1e-8)

    # k-means++ initialization
    indices = []
    first = torch.randint(0, N, (1,), device=x.device)
    indices.append(first.item())
    for _ in range(1, k):
        centers = x[torch.tensor(indices, device=x.device)]
        sim = x @ centers.t()
        closest_sim, _ = sim.max(dim=1)
        dist = 1 - closest_sim.clamp(-1, 1)
        probs = dist / dist.sum()
        next_idx = torch.multinomial(probs, 1)
        indices.append(next_idx.item())

    centroids = x[torch.tensor(indices, device=x.device)]

    # Balanced assignment capacities
    base_cap = N // k
    remainder = N % k
    capacities = torch.full((k,), base_cap, device=x.device, dtype=torch.long)
    if remainder > 0:
        capacities[:remainder] += 1

    prev_cluster_ids = None
    prev_loss = None
    loss = None

    for iter_num in range(num_iters):
        sim = x @ centroids.t()
        dists = 1.0 - sim.clamp(-1.0, 1.0)

        cluster_ids = torch.full((N,), -1, device=x.device, dtype=torch.long)
        remaining_cap = capacities.clone()

        _, sorted_clusters = torch.sort(dists, dim=1)

        for rank in range(k):
            unassigned = cluster_ids.eq(-1)
            if not unassigned.any():
                break

            cand_clusters = sorted_clusters[unassigned, rank]
            unassigned_idx = unassigned.nonzero(as_tuple=False).squeeze(1)

            for j in range(k):
                if remaining_cap[j] <= 0:
                    continue

                want_j_mask = cand_clusters.eq(j)
                if not want_j_mask.any():
                    continue

                cand_indices = unassigned_idx[want_j_mask]
                take = min(remaining_cap[j].item(), cand_indices.numel())
                if take <= 0:
                    continue

                chosen = cand_indices[:take]
                cluster_ids[chosen] = j
                remaining_cap[j] -= take

        if (cluster_ids == -1).any():
            raise RuntimeError("Balanced k-means assignment failed: some points unassigned")

        if prev_cluster_ids is not None and torch.equal(cluster_ids, prev_cluster_ids):
            break

        point_sim = sim[torch.arange(N, device=x.device), cluster_ids]
        point_dists = 1.0 - point_sim.clamp(-1.0, 1.0)
        loss = point_dists.mean().item()

        if prev_loss is not None and loss is not None:
            if abs(loss - prev_loss) < 1e-6:
                break

        prev_cluster_ids = cluster_ids.clone()
        prev_loss = loss

        # Update centroids
        new_centroids = torch.zeros_like(centroids)
        for j in range(k):
            mask = cluster_ids == j
            if mask.any():
                new_centroids[j] = x[mask].mean(dim=0)
            else:
                rand_idx = torch.randint(0, N, (1,), device=x.device)
                new_centroids[j] = x[rand_idx]

        centroids = F.normalize(new_centroids, p=2, dim=-1, eps=1e-8)

    return cluster_ids, centroids, loss

## Collect Neuron Features Per Subclass

In [ ]:
def _collect_neuron_features_per_subclass(model, tokenizer, neuron_masks, model_name, batch_size=5, save_path=None):
    """
    Collect neuron activation features grouped by subclass.
    
    Args:
        model: The circuit discovery model
        tokenizer: Tokenizer for decoding
        neuron_masks: Boolean masks for active neurons per subclass
        model_name: Name of the base model
        batch_size: Batch size for processing
        save_path: Optional path to save features
    
    Returns:
        features_per_subclass: Dict mapping subclass -> feature tensor
        indices_per_subclass: Dict mapping subclass -> neuron indices
    """
    activations_generator = NeuronActivationsGenerator(model_name, batch_size=batch_size)
    num_batches = (activations_generator.ids.shape[0] + batch_size - 1) // batch_size

    k_classes = neuron_masks.size(0)

    indices_per_subclass = {}
    for c in range(k_classes):
        mask = neuron_masks[c]
        idx = torch.nonzero(mask, as_tuple=False).squeeze(1)
        if idx.numel() == 0:
            continue
        indices_per_subclass[c] = idx

    features_lists = {c: [] for c in indices_per_subclass.keys()}

    for batch_idx in range(num_batches):
        out_fname = activations_generator.generate_batch_activations(batch_idx, log=True)
        batch = torch.load(out_fname, map_location="cpu")

        ids, activations_dict = batch["ids"], batch["activations"]

        if isinstance(ids, torch.Tensor):
            input_id_list = ids.tolist()
        else:
            input_id_list = ids

        prompts = tokenizer.batch_decode(input_id_list, skip_special_tokens=True)
        activations = _stack_layer_activations(activations_dict).to(device)

        op1, op2, res = parse_equation(prompts, device=device)
        classifier_logits = model.classify_problem(op1, op2, res)
        hard = F.gumbel_softmax(classifier_logits, tau=model.tau, dim=-1, hard=True)
        subclass = hard.argmax(dim=-1)

        mean_activations = activations.mean(dim=1)

        for c, idx in indices_per_subclass.items():
            ex_mask = subclass == c
            if not ex_mask.any():
                continue
            acts_c = mean_activations[ex_mask][:, idx]
            file_feature_c = acts_c.mean(dim=0)
            features_lists[c].append(file_feature_c)

    activations_generator.remove_handles()

    features_per_subclass = {}
    for c, feats in features_lists.items():
        if not feats:
            continue
        feats_tensor = torch.stack(feats, dim=0).to(device)
        features_per_subclass[c] = feats_tensor.t()

    if save_path is not None:
        torch.save(
            {
                "model_name": model_name,
                "features_per_subclass": {c: v.detach().cpu() for c, v in features_per_subclass.items()},
                "indices_per_subclass": {c: idx.detach().cpu() for c, idx in indices_per_subclass.items()},
            },
            save_path,
        )
        print(f"Saved subclass neuron features to {save_path}")

    return features_per_subclass, indices_per_subclass

## Run Neuron K-Means Clustering

In [ ]:
def run_neuron_kmeans(
    k,
    subclass: int,
    model,
    tokenizer,
    neuron_masks,
    model_name,
    batch_size=5,
    num_iters=100,
    log=True,
    subclass_features_path=None,
):
    """
    Run k-means clustering on neurons for a specific subclass.
    
    Args:
        k: Number of clusters
        subclass: Which subclass to cluster
        model: The circuit discovery model
        tokenizer: Tokenizer
        neuron_masks: Boolean masks for active neurons
        model_name: Model name
        batch_size: Batch size for feature collection
        num_iters: Max k-means iterations
        log: Whether to print progress
        subclass_features_path: Path to cached features
    
    Returns:
        cluster_ids: Cluster assignments
        centroids: Cluster centroids
        loss: Final clustering loss
    """
    safe = _safe_model_name(model_name)
    results_dir = os.path.join("results", "neuron-clustering", safe)
    os.makedirs(results_dir, exist_ok=True)

    if subclass_features_path is None:
        subclass_features_path = os.path.join(results_dir, "subclass_features.pt")

    if subclass_features_path is not None and os.path.exists(subclass_features_path):
        ckpt = torch.load(subclass_features_path, map_location=device)
        features_per_subclass = {int(c): v.to(device) for c, v in ckpt["features_per_subclass"].items()}
        indices_per_subclass = {int(c): idx.to(device) for c, idx in ckpt["indices_per_subclass"].items()}
        print(f"Loaded cached features from {subclass_features_path}")
    else:
        features_per_subclass, indices_per_subclass = _collect_neuron_features_per_subclass(
            model, tokenizer, neuron_masks, model_name,
            batch_size=batch_size, save_path=subclass_features_path
        )

    if subclass not in features_per_subclass:
        raise ValueError(f"No features found for subclass {subclass}")

    x = features_per_subclass[subclass]
    subclass_indices = indices_per_subclass[subclass]

    cluster_ids, centroids, loss = _kmeans_cosine(x, k=k, num_iters=num_iters)

    cluster_to_indices = {}
    for j in range(k):
        mask = cluster_ids == j
        if mask.any():
            cluster_to_indices[j] = subclass_indices[mask].cpu()
        else:
            cluster_to_indices[j] = torch.empty(0, dtype=subclass_indices.dtype)

    # Save results
    clusters_path = os.path.join(results_dir, f"subclass_{subclass}_clusters", f"k{k}.pt")
    os.makedirs(os.path.dirname(clusters_path), exist_ok=True)
    torch.save(
        {
            "model_name": model_name,
            "subclass": subclass,
            "k": k,
            "cluster_ids": cluster_ids.cpu(),
            "subclass_indices": subclass_indices.cpu(),
            "cluster_to_indices": cluster_to_indices,
            "loss": loss,
        },
        clusters_path,
    )

    if log:
        print(f"Subclass {subclass}: k-means over neurons completed.")
        print(f"Mean cosine distance to centroids (loss): {loss:.6f}")
        for j in range(k):
            size = int((cluster_ids == j).sum().item())
            print(f"  Cluster {j}: size={size}")
        print(f"Saved cluster assignments to {clusters_path}")

    return cluster_ids, centroids, loss

## Run Clustering on a Single Subclass

Example: cluster neurons for subclass 0 with k=7 clusters.

In [ ]:
# Run clustering for a single subclass
subclass = 0  # Change this to cluster different subclasses
k = 7  # Number of clusters

if neuron_masks[subclass].any().item():
    cluster_ids, centroids, loss = run_neuron_kmeans(
        k=k,
        subclass=subclass,
        model=model,
        tokenizer=tokenizer,
        neuron_masks=neuron_masks,
        model_name=model_name,
        batch_size=5,
        num_iters=100,
        log=True
    )
else:
    print(f"Subclass {subclass} has no active neurons")

## Grid Search Over k Values

Run k-means for multiple values of k to find optimal clustering.

In [ ]:
# Grid search over k values for all subclasses
k_range = range(2, 10)  # Test k from 2 to 9
k_gs_results = {}

for subclass in range(k_classes):
    if neuron_masks[subclass].any().item():
        print(f"\nProcessing subclass {subclass}")
        k_gs_results[subclass] = {}
        for k in k_range:
            try:
                _, _, loss = run_neuron_kmeans(
                    k=k,
                    subclass=subclass,
                    model=model,
                    tokenizer=tokenizer,
                    neuron_masks=neuron_masks,
                    model_name=model_name,
                    log=False
                )
                k_gs_results[subclass][k] = loss
                print(f"  k={k}, loss={loss:.6f}")
            except ValueError as e:
                print(f"  k={k} failed: {e}")
    else:
        print(f"\nSkipping subclass {subclass} (no active neurons)")

## Save Grid Search Results

In [ ]:
# Save grid search results to JSON
safe = _safe_model_name(model_name)
results_dir = os.path.join("results", "neuron-clustering", safe)
os.makedirs(results_dir, exist_ok=True)

out_path = os.path.join(results_dir, "k_gs_results.json")

# Convert to JSON-serializable format
k_gs_json = {str(k): {str(kk): v for kk, v in vv.items()} for k, vv in k_gs_results.items()}

with open(out_path, "w") as f:
    json.dump(k_gs_json, f, indent=2)

print(f"Saved grid search results to {out_path}")

## Visualize Clustering Results (Optional)

In [ ]:
import matplotlib.pyplot as plt

# Plot loss vs k for each subclass
fig, ax = plt.subplots(figsize=(10, 6))

for subclass, results in k_gs_results.items():
    ks = sorted(results.keys())
    losses = [results[k] for k in ks]
    ax.plot(ks, losses, marker='o', label=f'Subclass {subclass}')

ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Mean Cosine Distance (Loss)')
ax.set_title('Neuron Clustering: Loss vs Number of Clusters')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'k_vs_loss.png'), dpi=150)
plt.show()